# Test Notebook

Smoke test for running Jupyter notebooks against the local `fantasy-player-valuation` environment.

In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd()
if repo_root.name == "analysis":
    repo_root = repo_root.parent

src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(sys.executable)
print(repo_root)

C:\Users\limic\miniconda3\envs\fantasy-player-valuation\python.exe
C:\dev\fantasy_player_valuation


In [2]:
import ffvaluation

print(ffvaluation.__file__)

C:\dev\fantasy_player_valuation\src\ffvaluation\__init__.py


In [3]:
import sqlite3

db_path = repo_root / "data" / "raw" / "sleeper" / "discovery" / "discovery.sqlite"
print(db_path)

if db_path.exists():
    with sqlite3.connect(db_path) as con:
        counts = {
            table: con.execute(f"select count(*) from {table}").fetchone()[0]
            for table in ["users", "frontier", "leagues", "league_users"]
        }
    counts
else:
    "discovery.sqlite not found"

C:\dev\fantasy_player_valuation\data\raw\sleeper\discovery\discovery.sqlite


In [7]:
def find_sleeper_user(term: str, db_path=db_path):
    """Search discovery users and frontier rows by user id, username, or display name."""
    normalized = term.strip().lower()
    if not normalized:
        raise ValueError("term must not be blank")
    if not db_path.exists():
        raise FileNotFoundError(db_path)

    with sqlite3.connect(db_path) as con:
        con.row_factory = sqlite3.Row
        users = con.execute(
            """
            select user_id, display_name
            from users
            where lower(user_id) = ? or lower(display_name) = ?
            order by display_name, user_id
            """,
            (normalized, normalized),
        ).fetchall()
        frontier = con.execute(
            """
            select user_id, username, display_name, discovered_at,
                   discovered_from_league_id, expanded_at
            from frontier
            where lower(user_id) = ?
               or lower(coalesce(username, '')) = ?
               or lower(coalesce(display_name, '')) = ?
            order by discovered_at, user_id
            """,
            (normalized, normalized, normalized),
        ).fetchall()

    return {
        "users": [dict(row) for row in users],
        "frontier": [dict(row) for row in frontier],
    }


find_sleeper_user("bbroc")

{'users': [{'user_id': '466725566741999616', 'display_name': 'bbroc'}],
 'frontier': [{'user_id': '466725566741999616',
   'username': None,
   'display_name': 'bbroc',
   'discovered_at': '2026-06-12T12:28:32.925154+00:00',
   'discovered_from_league_id': '1335078674310926336',
   'expanded_at': '2026-06-16T12:40:48.173512+00:00'}]}